In [ ]:
from torch import nn
import torch.nn.functional as F

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels, num_heads=12, attn_p=0, proj_p=0):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.scale = self.head_dim ** -0.5

        self.query = nn.Linear(in_channels, in_channels)        # Diese Daten werden nach und nach angepasst, starten random und werden dann durch das Training immer besser. 
        self.value = nn.Linear(in_channels, in_channels)        # Es ist wichtig, dass die Dimensionen der Query, Key und Value gleich sind, damit wir die Attention berechnen können.
        self.key = nn.Linear(in_channels, in_channels)          

        self.attn_p = attn_p
        self.proj_p = nn.Linear(in_channels, in_channels)
        self.proj_drop = nn.Dropout(proj_p)
    
    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        q = self.query(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_p)
        
        x = x.transpose(1,2).reshape(batch_size, seq_len, embed_dim)
        x = self.proj_drop(self.proj_p(x))
        return x


In [5]:
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) Block.
    
    Warum ist dieses MLP so wichtig?
    Während die Self-Attention-Schicht dafür zuständig ist, Informationen 
    ZWISCHEN verschiedenen Pixeln oder Sequenzpositionen auszutauschen 
    (räumlicher Kontext), arbeitet das MLP isoliert auf jedem einzelnen Pixel. 
    
    Es nimmt die durch die Attention neu gesammelten Informationen (die Features/Kanäle) 
    eines Pixels und verarbeitet diese tiefgreifend in sich selbst. Das MLP mischt 
    also nur entlang der Feature-Dimension. 
    
    Durch die Expansion der Kanäle (meist das 2- bis 4-fache in der Mitte) und 
    die nicht-lineare Aktivierungsfunktion (GELU) erhält das Modell hier den 
    nötigen "Denkraum", um komplexe Muster zu speichern und die gesammelten 
    Erkenntnisse der Attention zu festigen.
    
    Zusammenfassung der Aufgabenteilung im Transformer:
    - Attention = Informationsaustausch über das gesamte Bild (Kommunikation).
    - MLP = Tiefe Verarbeitung der gesammelten Informationen pro Pixel (Einzelarbeit).
    """
    def __init__(self, in_channels, mlp_ratio=4, mlp_p=0):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels * mlp_ratio)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(mlp_p)
        self.fc2 = nn.Linear(in_channels * mlp_ratio, in_channels)
        self.drop2 = nn.Dropout(mlp_p)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x= self.fc2(x)
        x = self.drop2(x)
        return x
    


In [ ]:
class ImageTransformerBlock(nn.Module):
    def __init__(self, in_channels, num_heads=4, mlp_ratio=2, proj_p=0, attn_p=0, mlp_p=0):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(in_channels, eps=1e-6)
        self.attn = SelfAttention(in_channels=in_channels,
                                  num_heads=num_heads, 
                                  attn_p=attn_p,
                                  proj_p=proj_p)
        
        self.norm2 = nn.LayerNorm(in_channels, eps=1e-6)
        self.mlp = MLP(in_channels=in_channels,
                       mlp_ratio=mlp_ratio,
                       mlp_p=mlp_p)
        
    def forward(self, x):
        batch_size, channels, height, width = x.shape
      
        ### 1. Übersetzer: Bild in flache Sequenz umwandeln -> (Batch, H*W, Channels)
        x = x.reshape(batch_size, channels, height*width).permute(0,2,1)
        
        ### 2. Attention anwenden
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))

        ### 3. Übersetzer: Sequenz wieder zu einem Bild zusammenfalten -> (Batch, Channels, H, W)
        x = x.permute(0,2,1).reshape(batch_size, channels, height, width)
        return x

In [ ]:
class PointTransformerBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 num_heads=4, 
                 mlp_ratio=2,
                 proj_p=0,
                 attn_p=0,
                 mlp_p=0):
        
        super().__init__()
        
        # 1. Self-Attention (Punkte reden mit Punkten, um die Form zu glätten)
        self.norm1 = nn.LayerNorm(in_channels, eps=1e-6)
        self.attn = SelfAttention(in_channels=in_channels,
                                  num_heads=num_heads, 
                                  attn_p=attn_p,
                                  proj_p=proj_p)
        
        
        # 2. MLP (Punkte verarbeiten ihre Position und die Bild-Features isoliert)
        self.norm2 = nn.LayerNorm(in_channels, eps=1e-6)
        self.mlp = MLP(in_channels=in_channels,
                       mlp_ratio=mlp_ratio,
                       mlp_p=mlp_p)
        
    def forward(self, x):
        # x hat hier schon die Form (Batch, n_punkte, in_channels)
        # Und ganz wichtig: 'x' enthält hier bereits die eingesaugten Bild-Features!
        
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))

        return x